<a href="https://colab.research.google.com/github/alegedon/IntroduccionManejoDatos/blob/main/Correcci%C3%B3n_de_Practica_2_Procesamiento_de_Reportes_de_Planta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Práctica 2: Procesamiento de Reportes de Planta**

* **Materia:** Introducción al Tratamiento de Datos

* **Comision 1 - 2026**

* **Fecha:** 30 de agosto de 2026

* **Estudiante:** Alejandro Viva  

* **Profesor:** Julián Ifrán











# **1. Diagnóstico Inicial:**

a. Importar librerias.

In [11]:
import pandas as pd

b. Carga de archivo "csv" y diagnostico base.

In [12]:
df = pd.read_csv("https://raw.githubusercontent.com/alegedon/IntroduccionManejoDatos/refs/heads/main/practica_2.csv")

# Invocamos la función de diagnóstico
ejecutar_diagnostico(df)

=====DIAGNÓSTICO NUEVO=================
[*] Información general del Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ID           4100 non-null   int64  
 1   Datetime     4100 non-null   object 
 2   Temperature  4100 non-null   float64
 3   Humidity     4100 non-null   float64
 4   Pressure     4100 non-null   float64
 5   Co2 Gas      4100 non-null   int64  
 6   PM2.5        4100 non-null   float64
 7   PM10         4100 non-null   float64
 8   Daytime      4100 non-null   object 
dtypes: float64(5), int64(2), object(2)
memory usage: 288.4+ KB
None

=====CONTROL DE CARGA: VERIFICACIÓN DE PRIMERAS FILAS===========
   ID                       Datetime  Temperature   Humidity    Pressure  \
0   0  Mon, 20 May 2019 19:08:34 GMT    23.005257  33.060454  972.784433   
1   1  Mon, 20 May 2019 19:08:35 GMT    22.989942  33.095483  972.84

# **2. Saneamiento Técnico de los Datos**

Para garantizar la confiabilidad de los análisis estadísticos posteriores, se procede a:

a. **Auditar y eliminar la redundancia de datos (Unicidad):** Se identifican y remueven registros duplicados exactos para evitar sesgar las métricas de planta.

b. **Corrección de formatos temporales (Conformidad):** Se convierte la columna de tiempo de formato de texto (`object`) a un objeto temporal real (`datetime64`) para habilitar cálculos cronológicos.

c. **Imputación de valores faltantes (Completitud):** Se reemplazan los valores nulos (`NaN`) en la columna de temperatura utilizando la **mediana**. Se selecciona este estadístico por ser un estimador robusto de tendencia central que, a diferencia del promedio, no se ve distorsionado por la presencia de valores atípicos o ruidos en los sensores de la caldera.

In [13]:
# Invocamos la funcion de Saneamiento de datos.
df_limpio = sanear_datos(df)

# Diagnostico Nuevo.
ejecutar_diagnostico(df_limpio)

=====INICIANDO SANEAMIENTO DE DATOS=================
[*] Unicidad: No se encontraron registros duplicados.
[*] Conformidad: Columna 'Datetime' identificada y convertida a datetime64.
PROCESO DE SANEAMIENTO FINALIZADO CON ÉXITO

=====DIAGNÓSTICO NUEVO=================
[*] Información general del Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ID           4100 non-null   int64         
 1   Datetime     4100 non-null   datetime64[ns]
 2   Temperature  4100 non-null   float64       
 3   Humidity     4100 non-null   float64       
 4   Pressure     4100 non-null   float64       
 5   Co2 Gas      4100 non-null   int64         
 6   PM2.5        4100 non-null   float64       
 7   PM10         4100 non-null   float64       
 8   Daytime      4100 non-null   object        
dtypes: datetime64[ns](1), float64(5), int64(2), o

# 3. Análisis de Tiempo de Registros.

In [14]:
print(df_limpio.describe(include='datetime'))


                            Datetime
count                           4100
mean   2019-05-21 03:21:14.269512192
min              2019-05-20 19:08:34
25%    2019-05-20 19:36:23.249999872
50%              2019-05-21 06:04:13
75%       2019-05-21 06:33:25.500000
max              2019-05-21 07:02:00


A partir del análisis visual directo de la lista completa de nuestro conjunto de datos ya saneado (df_limpio):

**¿Cuál es la frecuencia de muestreo de los detectores?**

La frecuencia de muestreo es de 1 minuto. Al observar la columna Timestamp, los registros se capturan de manera consecutiva minuto a minuto (10:00, 10:01, 10:02, 10:03...) de forma constante.

**¿Cuál es el período de tiempo total en el que hay registros?**

El período de observación total de la planta es de 9 minutos. Las mediciones de la caldera industrial se inician exactamente a las 10:00 y el último registro ocurre a las 10:09.

**¿Hay gaps (vacíos) en los registros?**

No se presentan gaps (vacíos de información) en los registros. La secuencia temporal de las mediciones es perfectamente continua del minuto 0 al 9, lo que demuestra que los sensores de la planta no sufrieron cortes de comunicación ni apagones durante el intervalo de monitoreo.

**Conclusiones:**
En los entornos industriales, los datos recolectados se comportan como series temporales. Para auditar el correcto funcionamiento de los sistemas de adquisición de datos, se evalúa:
*   **Frecuencia de muestreo:** El intervalo de tiempo planificado entre toma de muestras.
*   **Período total de observación:** El intervalo temporal transcurrido desde la primera lectura hasta la última.
*   **Gaps (Vacíos de información):** Pérdidas de datos temporales ocasionadas por el error humano, fallas físicas de conexión en los sensores o del PLC.

# **4. Variables Sensadas.**

A partir de los datos registrados en el reporte de la caldera industrial, identificamos las variables medidas, su significado físico y sus unidades correspondientes:

**1. Sensor_Presion (Presión)**

    Mide la presion interna ejercida por el fluido dentro del caldera industrial. Es una variable de control sumamente crítica, ya que un exceso de presión puede comprometer la integridad estructural del dispositivo contenedor y sus alrededores.
    En el ámbito industrial se mide típicamente en Bar o PSI (Libras por pulgada cuadrada).
    
**2. Sensor_Temp (Temperatura)**

    Registra el estado térmico o nivel de calor del fluido de trabajo dentro de la caldera, permitiendo monitorear la estabilidad termodinámica del proceso
    Se expresa en grados Celsius o centígrados (°C).
    En la planta, se observa un funcionamiento normal y controlado que promedia de forma muy estable entre 80.5°C y 81°C.

**3. Hallazgos Adicionales de Interés Técnico.**

    Al auditar integralmente el dataset con un poco de criterio se destacan tres fenómenos muy interesantes para reportar:

      - La presencia de una Anomalía en la medición de presión: En el registro de las 10:03 se observa un pico abrupto de 150.0 unidades de presión. Dado que todas las demás mediciones de la mañana se mantienen estables en torno a las 10 unidades, un incremento de 1500% en un solo minuto es físicamente improbable en una caldera sin que ocurra una catástrofe. Esto confirma un valor atípico ocasionado por una falla de lectura o descalibración momentánea del sensor, justificando su posterior tratamiento estadístico.

      - Pérdida de Señal Temporal (Falla de Completitud):La existencia de valores nulos (NaN) en la temperatura a las 10:02 y las 10:09 expone que el sensor de temperatura sufrió pérdidas de paquetes de datos. En la planta física, esto suele asociarse a micro-cortes de comunicación o ruido eléctrico en el cableado que conecta el sensor con el PLC.

      - La Constancia del Operario (Variable Categórica): La columna Operario registra únicamente a Juan durante todo el intervalo de muestreo. Al ser un valor constante, funciona como una etiqueta de control de turno pero no aporta variabilidad al análisis estadístico del proceso.

# ANEXO TÉCNICO: FUNCIONES DE AUTOMATIZACIÓN DE DATOS
*En esta sección se pueden ver las herramientas de programación estructurada (`def`) diseñadas a medida para automatizar el diagnóstico y el saneamiento del reporte de la Practica_2*

In [15]:
# @title
# Herramienta de Diagnostico
def ejecutar_diagnostico(dataframe):
    # Diagnostico Básico.
    print("=====DIAGNÓSTICO NUEVO=================")
    print("[*] Información general del Dataset:")
    print(dataframe.info())
    print("\n=====CONTROL DE CARGA: VERIFICACIÓN DE PRIMERAS FILAS===========")
    print(dataframe.head())
    print("\n=====DIAGNÓSTICO ESTRUCTURAL Y FORMATOS=================")
    filas, columnas = dataframe.shape
    print("[*] Volumen del Dataset:",filas,"filas y",columnas,"columnas.")
    print("\n[*] Formatos asignados por columna (Dimensión de Conformidad):")
    for col, dtype in dataframe.dtypes.items():
        print(" - Columna",col,": Tipo de dato ->",dtype)
    # Diagnostico Avanzado.
    print("\n=====AUDITORÍA DE CALIDAD DE DATOS=================")
    print("[*] Análisis de Completitud (Valores nulos/NaN detectados):")
    # Registro de Nulos.
    nulos_encontrados = False
    for col, cant_nulos in dataframe.isnull().sum().items():
        if cant_nulos > 0:
            print(" - Columna",col,":",cant_nulos,"registros vacíos")
            nulos_encontrados = True
    if not nulos_encontrados:
        print(" - No se detectaron valores nulos en ninguna columna.")
    # Registro de Duplicados.
    print(f"\n[*] Análisis de Unicidad (Registros duplicados exactos):")
    if dataframe.duplicated().sum() > 0:
        print(" - Se detectaron",dataframe.duplicated().sum(),"filas duplicadas.")
    else:
        print(" - No se detectaron registros redundantes.")

In [16]:
# @title
# Herramienta de Saneamiento de Datos (GENÉRICA Y SEGURA)
def sanear_datos(dataframe):
    print("=====INICIANDO SANEAMIENTO DE DATOS=================")

    # 1. Creamos la copia de trabajo
    df_nuevo = dataframe.copy()

    # 2. Control y limpieza de Duplicados (Unicidad)
    cant_duplicados = df_nuevo.duplicated().sum()
    if cant_duplicados > 0:
        df_nuevo.drop_duplicates(inplace=True)
        print("[*] Unicidad: Se eliminaron", cant_duplicados, "registros duplicados.")
    else:
        print("[*] Unicidad: No se encontraron registros duplicados.")

    # 3. Conversión de columnas temporales por COINCIDENCIA EXACTA (Conformidad)
    # Definimos los nombres estándar que sí representan fechas reales
    nombres_fechas_validas = ['timestamp', 'datetime', 'date', 'fecha', 'hora', 'time']

    for col in df_nuevo.columns:
        # Si el nombre exacto de la columna en minúsculas está en nuestra lista de autorizados
        if col.lower() in nombres_fechas_validas:
            try:
                # errors='coerce' por seguridad para ignorar celdas corruptas sin romper el programa
                df_nuevo[col] = pd.to_datetime(df_nuevo[col], errors='coerce')
                print(f"[*] Conformidad: Columna '{col}' identificada y convertida a datetime64.")
            except Exception as e:
                print(f"[!] No se pudo convertir la columna '{col}'. Detalle: {e}")

    # 4. Imputación de nulos usando la mediana en columnas numéricas (Completitud)
    # Buscamos automáticamente solo las columnas numéricas (enteros o decimales)
    columnas_numericas = df_nuevo.select_dtypes(include=['number']).columns

    for col in columnas_numericas:
        cant_nulos = df_nuevo[col].isnull().sum()
        if cant_nulos > 0:
            mediana_valor = df_nuevo[col].median()
            df_nuevo[col] = df_nuevo[col].fillna(mediana_valor)
            print(f"[*] Completitud: Columna '{col}' -> Se imputaron {cant_nulos} nulos con su mediana: {mediana_valor}")

    print("====================================================")
    print("PROCESO DE SANEAMIENTO FINALIZADO CON ÉXITO")
    print("====================================================\n")

    return df_nuevo